In [ ]:
# Étape 1: Importation des librairies et chargement du dataset
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.tree import plot_tree

# Chargement du dataset
data = load_breast_cancer()

# Exploration du dataset
X = data.data  # Les features
y = data.target  # Les labels (benign = 0, malignant = 1)

# Affichage des informations sur le dataset
print(f"Nombre d'observations: {X.shape[0]}")
print(f"Nombre de variables: {X.shape[1]}")
print(f"Classes: {np.unique(y)}")

In [ ]:
# Étape 2: Division du dataset en train et test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

In [ ]:
# Étape 3: Création du modèle de base
model = DecisionTreeClassifier(random_state=42)

In [ ]:
# Étape 4: Définition de la grille de recherche pour les hyperparamètres
param_grid = {
    'criterion': ['gini', 'entropy'],  # Critère de décision
    'max_depth': [2, 4, 6, 8, 10, None],  # Profondeur maximale
    'min_samples_split': [2, 5, 10],  # Nombre minimum d'échantillons pour splitter un nœud
    'min_samples_leaf': [1, 2, 4]  # Nombre minimum d'échantillons dans une feuille
}

In [ ]:
# Étape 5: Lancement du GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=2, scoring='accuracy', refit=True, verbose=1)
grid_search.fit(X_train, y_train)

In [ ]:
# Étape 6: Affichage des meilleurs paramètres et du score moyen de la validation croisée
print(f"Meilleurs paramètres: {grid_search.best_params_}")
print(f"Meilleur score de validation croisée: {grid_search.best_score_}")

In [ ]:
# Étape 7: Évaluation sur le dataset de test
best_model = grid_search.best_estimator_  # Utilisation du modèle optimal
y_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy sur le test: {test_accuracy:.4f}")

In [ ]:
# Étape 8: Visualisation de l'arbre de décision optimal
import matplotlib.pyplot as plt
plt.figure(figsize=(15,10))
plot_tree(best_model, filled=True, feature_names=data.feature_names, class_names=data.target_names)
plt.title("Arbre de décision optimal")
plt.show()

Pièges à éviter et corrections :

Ne pas stratifier la variable cible dans train_test_split() : Cela peut entraîner une distribution non équilibrée des classes entre l'entraînement et le test.

Ne pas utiliser un random_state dans DecisionTreeClassifier : Cela rend le modèle non reproductible. Il est important de fixer une valeur pour random_state afin d’obtenir les mêmes résultats à chaque exécution.

Utiliser un max_depth trop élevé ou None sans validation : Cela peut conduire à un modèle trop complexe (sur-apprentissage). Il faut tester plusieurs valeurs pour max_depth.

Ne pas tester les autres hyperparamètres : Ne se concentrer que sur max_depth peut limiter la performance. Il faut tester aussi min_samples_split, min_samples_leaf, et criterion.

Ne pas utiliser de validation croisée dans GridSearchCV : Cela peut entraîner une évaluation biaisée des modèles. cv=2 est le minimum pour éviter cela.

Ne pas utiliser un critère de scoring pertinent : Par défaut, GridSearchCV utilise accuracy, mais si l’on a un problème de classes déséquilibrées, il peut être plus pertinent de choisir un autre critère comme f1 ou roc_auc.

Ne pas visualiser l'arbre pour comprendre les décisions : L’arbre de décision peut être trop complexe pour des paramètres de profondeur élevés. Utiliser la visualisation permet de comprendre pourquoi certaines décisions sont prises par le modèle.

Réponses aux questions :

Effet de max_depth et min_samples_leaf sur le modèle :

Quand max_depth=None et min_samples_leaf=1, l'arbre devient très complexe et peut "mémoriser" les données d'entraînement (overfitting). Le modèle sera excellent sur les données d’entraînement, mais ne généralisera pas bien sur de nouvelles données.

Problème avec min_samples_split=1 et min_samples_leaf=1 :

Ces valeurs entraînent un modèle très flexible, trop ajusté aux données d’entraînement, ce qui peut amener à de l’overfitting. Cela rend l'arbre trop détaillé et non généralisable.

Trade-off entre biais et variance :

Biais (sous-apprentissage) survient lorsque l'arbre est trop simple (petit max_depth, grands min_samples_split et min_samples_leaf), et le modèle échoue à capturer les tendances sous-jacentes des données.

Variance (sur-apprentissage) survient lorsque l’arbre est trop complexe (grand max_depth, petits min_samples_split et min_samples_leaf), ce qui conduit à un modèle qui apprend trop bien les données d’entraînement mais échoue sur de nouvelles données.

Conclusion

Trop de complexité (haute profondeur + petits échantillons dans les feuilles) mène à de l’overfitting.

Trop de simplicité (basse profondeur + grands échantillons dans les feuilles) mène à du sous-apprentissage.

Ainsi, le but est de trouver un bon compromis entre biais et variance en ajustant soigneusement les hyperparamètres.

Si vous avez des questions ou souhaitez ajuster des paramètres supplémentaires, n'hésitez pas à me le faire savoir !


mets moi tout ça en markdown


In [ ]:
pip install -U scikit-learn



In [ ]:
Exercice 2

In [ ]:
# Étape 1: Importation des librairies nécessaires
import time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
from sklearn.metrics import plot_confusion_matrix

# Étape 2: Division du dataset en train et test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)



In [ ]:
# Étape 3: Définition de la grille de recherche pour les hyperparamètres de SVM
svm_param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

# Création du modèle SVM
svm_model = SVC(random_state=42)

# GridSearchCV pour SVM
svm_grid_search = GridSearchCV(svm_model, svm_param_grid, cv=3, scoring='f1', refit=True, verbose=1)
start_time_svm = time.time()
svm_grid_search.fit(X_train, y_train)
end_time_svm = time.time()

# Calcul du temps d'inférence pour SVM
svm_inference_time = end_time_svm - start_time_svm

# Affichage des meilleurs paramètres et du meilleur score pour SVM
print(f"Meilleurs paramètres SVM: {svm_grid_search.best_params_}")
print(f"Meilleur F1-score SVM sur le train: {svm_grid_search.best_score_}")

In [ ]:
# Étape 4: Définition de la grille de recherche pour les hyperparamètres de Decision Tree
dt_param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [2, 4, 6, 8, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Création du modèle Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)

# GridSearchCV pour Decision Tree
dt_grid_search = GridSearchCV(dt_model, dt_param_grid, cv=3, scoring='f1', refit=True, verbose=1)
start_time_dt = time.time()
dt_grid_search.fit(X_train, y_train)
end_time_dt = time.time()

# Calcul du temps d'inférence pour Decision Tree
dt_inference_time = end_time_dt - start_time_dt

# Affichage des meilleurs paramètres et du meilleur score pour Decision Tree
print(f"Meilleurs paramètres Decision Tree: {dt_grid_search.best_params_}")
print(f"Meilleur F1-score Decision Tree sur le train: {dt_grid_search.best_score_}")

In [ ]:
# Étape 5: Évaluation sur le dataset de test
svm_best_model = svm_grid_search.best_estimator_
dt_best_model = dt_grid_search.best_estimator_

# Prédictions et évaluation des modèles
svm_y_pred = svm_best_model.predict(X_test)
dt_y_pred = dt_best_model.predict(X_test)

svm_f1 = f1_score(y_test, svm_y_pred)
dt_f1 = f1_score(y_test, dt_y_pred)

print(f"F1-score SVM sur le test: {svm_f1}")
print(f"F1-score Decision Tree sur le test: {dt_f1}")

# Affichage de la matrice de confusion
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

# Matrice de confusion pour SVM
plot_confusion_matrix(svm_best_model, X_test, y_test, ax=ax[0], cmap=plt.cm.Blues)
ax[0].set_title("Matrice de confusion SVM")

# Matrice de confusion pour Decision Tree
plot_confusion_matrix(dt_best_model, X_test, y_test, ax=ax[1], cmap=plt.cm.Blues)
ax[1].set_title("Matrice de confusion Decision Tree")

plt.show()

# Comparaison des temps d'inférence
print(f"Temps d'inférence SVM: {svm_inference_time:.4f} secondes")
print(f"Temps d'inférence Decision Tree: {dt_inference_time:.4f} secondes")